In [7]:
import torch
import os

# Paths to your checkpoints
path_1 = "/l/users/yohan.abeysinghe/pangu_data/results/pm_4_29/models/model_weights_1.pth"
path_2 = "/l/users/yohan.abeysinghe/pangu_data/results/pm_4_29/models/model_weights_2.pth"

# Load both state_dicts
sd1 = torch.load(path_1, map_location='cpu')
sd2 = torch.load(path_2, map_location='cpu')

# If stored with "model" key
if 'model' in sd1:
    sd1 = sd1['model']
if 'model' in sd2:
    sd2 = sd2['model']

# Compare the keys
assert sd1.keys() == sd2.keys(), "State dicts have different keys!"

# Find changed weights
changed = []
unchanged = []
for k in sd1:
    if not torch.equal(sd1[k], sd2[k]):
        changed.append(k)
    else:
        unchanged.append(k)

print(f"Changed parameters: {len(changed)}")
print(f"Unchanged parameters: {len(unchanged)}")
print("\nSome changed parameter names:")
for name in changed[:10]:
    print(name)

Changed parameters: 138
Unchanged parameters: 219

Some changed parameter names:
base_model.model._input_layer.conv_surface.weight
base_model.model._input_layer.conv_surface.bias
base_model.model.downsample.linear.lora_A.default.weight
base_model.model.downsample.linear.lora_B.default.weight
base_model.model.layers.EarthSpecificLayer0.blocks.EarthSpecificBlock0.linear.linear1.lora_A.default.weight
base_model.model.layers.EarthSpecificLayer0.blocks.EarthSpecificBlock0.linear.linear1.lora_B.default.weight
base_model.model.layers.EarthSpecificLayer0.blocks.EarthSpecificBlock0.linear.linear2.lora_A.default.weight
base_model.model.layers.EarthSpecificLayer0.blocks.EarthSpecificBlock0.linear.linear2.lora_B.default.weight
base_model.model.layers.EarthSpecificLayer0.blocks.EarthSpecificBlock0.attention.linear1.lora_A.default.weight
base_model.model.layers.EarthSpecificLayer0.blocks.EarthSpecificBlock0.attention.linear1.lora_B.default.weight


In [8]:
import os
import logging
import torch
import importlib
import torch.nn as nn
from peft import LoraConfig, get_peft_model
from models.pangu_model import PanguModel

# Manually set config name and output directory
config_name = 'config4'
output_dir_name = 'pm_4_29'

# Load config module
config_module = importlib.import_module(f"configs.{config_name}")
cfg = config_module.cfg

In [9]:
import sys
sys.path.append("/home/yohan.abeysinghe/Pangu/pangu-pytorch")

from models.pangu_model import PanguModel
from peft import LoraConfig, get_peft_model

# Load model
model = PanguModel(device='cpu', cfg=cfg)  # Use correct config here

# Apply LoRA as before
target_modules = [n for n, m in model.named_modules() if isinstance(m, torch.nn.Linear)]
lora_config = LoraConfig(
    r=cfg.PG.TRAIN.Low_Rank,
    lora_alpha=16,
    target_modules=target_modules,
    lora_dropout=0.1,
    bias="none",
)
model = get_peft_model(model, lora_config)

# Get trainable params
trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print("\nTrainable parameters:")
for name in trainable:
    print(name)


Trainable parameters:
base_model.model.downsample.linear.lora_A.default.weight
base_model.model.downsample.linear.lora_B.default.weight
base_model.model.layers.EarthSpecificLayer0.blocks.EarthSpecificBlock0.linear.linear1.lora_A.default.weight
base_model.model.layers.EarthSpecificLayer0.blocks.EarthSpecificBlock0.linear.linear1.lora_B.default.weight
base_model.model.layers.EarthSpecificLayer0.blocks.EarthSpecificBlock0.linear.linear2.lora_A.default.weight
base_model.model.layers.EarthSpecificLayer0.blocks.EarthSpecificBlock0.linear.linear2.lora_B.default.weight
base_model.model.layers.EarthSpecificLayer0.blocks.EarthSpecificBlock0.attention.linear1.lora_A.default.weight
base_model.model.layers.EarthSpecificLayer0.blocks.EarthSpecificBlock0.attention.linear1.lora_B.default.weight
base_model.model.layers.EarthSpecificLayer0.blocks.EarthSpecificBlock0.attention.linear2.lora_A.default.weight
base_model.model.layers.EarthSpecificLayer0.blocks.EarthSpecificBlock0.attention.linear2.lora_B.de